In [23]:
import os
from dotenv import load_dotenv
import sys
from sqlalchemy import select, and_, or_, func, desc
from sqlalchemy.orm import aliased
from datetime import date, timedelta



In [42]:
sys.path.append("../src")

# 2. Ahora sí podemos importar tus modelos y la base de datos
from domain.db import Base, engine, Session
from domain.models import Region, Player, Team, PlayerProfile, TeamTournament, Tournament, Match


# 3. CREAR LAS TABLAS (La prueba de fuego)
# Esto lee tus clases y crea las tablas en dev.db
try:
    Base.metadata.create_all(bind=engine)
    print("✅ ¡Conexión exitosa y tablas creadas en dev.db!")
except Exception as e:
    print(f"❌ Error al crear las tablas: {e}")

✅ ¡Conexión exitosa y tablas creadas en dev.db!


In [3]:
with Session() as session:
    stmt = (
        select(Player)
        .where(
            and_(
                Player.active == True,
                or_(
                    Player.nickname.like("%a%"),
                    Player.nickname.like("%e%"),
                )
            )
            # and( x or( z / a ))
        )
        .order_by(Player.nickname)
    )

    players = session.scalars(stmt).all()


    for result in players:
        print(f"Player: {result.nickname}")

    print(result)

Player: ahess
Player: aimee66
Player: albert93
Player: alexandra04
Player: alishale
Player: allison90
Player: amanda84
Player: amandagreen
Player: amandahancock
Player: amy47
Player: andersonpenny
Player: andreabrown
Player: andrewparsons
Player: angelahoward
Player: anthonyriley
Player: antoniotorres
Player: ashley55
Player: ashley71
Player: benjamindavila
Player: benjaminwyatt
Player: bentonjennifer
Player: berryalexandra
Player: bgarcia
Player: bgonzalez
Player: blawrence
Player: blaze
Player: bortega
Player: brendajimenez
Player: browneric
Player: carol26
Player: carolcarey
Player: carolyndixon
Player: castanedabarry
Player: cbates
Player: cbutler
Player: cesar54
Player: cgallagher
Player: charles27
Player: charles79
Player: christianwatson
Player: christopher81
Player: christophergarcia
Player: cisneroserica
Player: colleen59
Player: cynthiaenglish
Player: cynthiahorn
Player: dakota10
Player: danabolton
Player: daniel23
Player: daniel43
Player: danielguerrero
Player: danny08
Playe

In [4]:
with Session() as session:
    stmt = (
        select(Player.nickname, PlayerProfile.total_wins)
        .outerjoin(PlayerProfile, Player.id_player == PlayerProfile.id_player_profile)
        .where(PlayerProfile.total_wins == (select(func.max(PlayerProfile.total_wins)).scalar_subquery()))
        # .group_by(Player.nickname)
        

        
    )

    rows = session.execute(stmt).all()


    for result in rows:
        print(f"Player: {result.nickname}, Total Wins: {result.total_wins}")

    # print(result)

Player: smithdanielle, Total Wins: 50


consultas

In [10]:
# Total points (Top 5 Teams)
stmt = (
    select(Team.name, func.sum(TeamTournament.points_obtained).label("total_points"))
    .join(TeamTournament, Team.id_team == TeamTournament.id_team)
    .group_by(Team.name)
    .order_by(desc("total_points"))
    .limit(5)
)

rows = session.execute(stmt).all()

for result in rows:
    print(f"Team: {result.name}, Total Points: {result.total_points}")

Team: red dragons, Total Points: 100
Team: blue phoenix, Total Points: 70
Team: Mckinney Ltd Esports, Total Points: 0
Team: Hughes, James and Roberts Esports, Total Points: 0
Team: Fuller-Lambert Esports, Total Points: 0


In [14]:
# Jugadores "Veteranos" por Región ( + 1 año de antigüedad)
hace_un_ano = date.today() - timedelta(days=365)

stmt = (
    select(Region.name, func.count(Player.id_player).label("player_count"))
    .join(Player, Region.id_region == Player.id_region)
    .where(Player.active == True, Player.registration_date < hace_un_ano)
    .group_by(Region.name)
)


rows = session.execute(stmt).all()

for result in rows:
    print(f"Region: {result.name}, Veteran Players: {result.player_count}")

Region: Bangladesh, Veteran Players: 2
Region: Venezuela, Veteran Players: 3
Region: Kiribati, Veteran Players: 1
Region: Cameroon, Veteran Players: 1
Region: Korea, Veteran Players: 1
Region: Jordan, Veteran Players: 1
Region: Cambodia, Veteran Players: 2
Region: Saint Helena, Veteran Players: 3
Region: Singapore, Veteran Players: 1
Region: San Marino, Veteran Players: 2
Region: Malta, Veteran Players: 1
Region: Colombia, Veteran Players: 3
Region: Grenada, Veteran Players: 1
Region: Saudi Arabia, Veteran Players: 1
Region: Cayman Islands, Veteran Players: 2
Region: Ukraine, Veteran Players: 2
Region: Holy See (Vatican City State), Veteran Players: 1
Region: Bouvet Island (Bouvetoya), Veteran Players: 1
Region: Cape Verde, Veteran Players: 2
Region: Israel, Veteran Players: 2
Region: Nauru, Veteran Players: 2
Region: Kenya, Veteran Players: 1
Region: Madagascar, Veteran Players: 1
Region: Benin, Veteran Players: 2
Region: Guinea, Veteran Players: 1
Region: Cyprus, Veteran Players: 1
R

In [32]:
# Detalle de Próximos Partidos (Con Nombres de Equipos)


from domain.models import Match

Team1 = aliased(Team)
Team2 = aliased(Team)

stmt = (
    select(
        Match.match_date,
        Team1.name.label("home_team"),
        Team2.name.label("away_team"),
        Tournament.name.label("tournament")
    )
    .join(Team1, Match.id_team_one == Team1.id_team)
    .join(Team2, Match.id_team_two == Team2.id_team)
    .join(Tournament, Match.id_tournament == Tournament.id_tournament)
    .where(Match.match_date >= date.today())
    .order_by(Match.match_date.asc())
)

rows = session.execute(stmt).all()

for result in rows:
    print(f"Date: {result.match_date}, Home Team: {result.home_team}, Away Team: {result.away_team}, Tournament: {result.tournament}")

Date: 2026-04-13 18:38:31.644649, Home Team: Torres PLC Esports, Away Team: Morrow, Edwards and Griffith Esports, Tournament: Friend Cup 100
Date: 2026-04-13 18:40:13.706812, Home Team: Burton, Schmidt and Hobbs Esports, Away Team: Collins-Thomas Esports, Tournament: Charge Cup 67
Date: 2026-04-13 18:40:13.708810, Home Team: Floyd, Jackson and Dennis Esports, Away Team: Wolfe-Lee Esports, Tournament: Statement Cup 135
Date: 2026-04-13 18:40:13.709138, Home Team: Shaw, Clark and Murphy Esports, Away Team: Holloway-Murphy Esports, Tournament: Consumer Cup 148
Date: 2026-04-14 18:38:31.640125, Home Team: Vance Inc Esports, Away Team: Nichols-Hayes Esports, Tournament: Mr Cup 1
Date: 2026-04-14 18:38:31.645903, Home Team: Armstrong-Roberts Esports, Away Team: Williams Inc Esports, Tournament: None Cup 147
Date: 2026-04-14 18:40:13.704691, Home Team: Burton, Schmidt and Hobbs Esports, Away Team: Stevens Group Esports, Tournament: Success Cup 17
Date: 2026-04-14 18:40:13.706543, Home Team: S

In [34]:
# Porcentaje de Victoria por Jugador (Win Rate)
from sqlalchemy import Float
from sqlalchemy import Float, cast



# Porcentaje de Victoria por Jugador (Win Rate)

stmt = (
    select(
        Player.nickname,
        PlayerProfile.total_wins,
        PlayerProfile.total_matches,
        (cast(PlayerProfile.total_wins, Float) / 
         cast(func.nullif(PlayerProfile.total_matches, 0), Float) * 100).label("win_rate")
    )
    .join(PlayerProfile, Player.id_player == PlayerProfile.id_player)
    .where(PlayerProfile.total_matches > 10) # Solo jugadores con experiencia
    .order_by(desc("win_rate"))
)

rows = session.execute(stmt).all()

for result in rows:
    print(f"Player: {result.nickname}, Total Wins: {result.total_wins}, Total Matches: {result.total_matches}, Win Rate: {result.win_rate}%")

Player: drewthompson, Total Wins: 43, Total Matches: 13, Win Rate: 330.7692307692308%
Player: michael31, Total Wins: 46, Total Matches: 15, Win Rate: 306.6666666666667%
Player: anthonyriley, Total Wins: 49, Total Matches: 18, Win Rate: 272.22222222222223%
Player: amy47, Total Wins: 46, Total Matches: 17, Win Rate: 270.5882352941177%
Player: kennethmarshall, Total Wins: 41, Total Matches: 16, Win Rate: 256.25%
Player: laurenthompson, Total Wins: 49, Total Matches: 21, Win Rate: 233.33333333333334%
Player: rmoore, Total Wins: 32, Total Matches: 14, Win Rate: 228.57142857142856%
Player: ashley55, Total Wins: 37, Total Matches: 17, Win Rate: 217.6470588235294%
Player: patriciaklein, Total Wins: 28, Total Matches: 13, Win Rate: 215.3846153846154%
Player: ahess, Total Wins: 48, Total Matches: 23, Win Rate: 208.69565217391303%
Player: jeffrey00, Total Wins: 23, Total Matches: 12, Win Rate: 191.66666666666669%
Player: robert00, Total Wins: 47, Total Matches: 25, Win Rate: 188.0%
Player: jesse0

In [41]:
# Estadísticas de "Pichichi" por Tipo de Juego

from domain.models import VideoGameType, PlayerTeam

stmt = (
    select(
        Region.name.label("region"),
        func.avg(PlayerProfile.total_wins).label("promedio_victorias")
    )
    .join(Player, Region.id_region == Player.id_region)
    .join(PlayerProfile, Player.id_player == PlayerProfile.id_player)
    .join(PlayerTeam, Player.id_player == PlayerTeam.id_player)
    .join(Team, PlayerTeam.id_team == Team.id_team)
    .join(TeamTournament, Team.id_team == TeamTournament.id_team)
    .join(Tournament, TeamTournament.id_tournament == Tournament.id_tournament)
    .join(VideoGameType, Tournament.id_video_game_type == VideoGameType.id_video_game_type)
    .where(VideoGameType.name == "MOBA") # O el tipo que quieras
    .group_by(Region.name)
)

rows = session.execute(stmt).all()
if not rows:
    print("No se encontraron resultados para el tipo de juego especificado.")
else:
    for result in rows:
        print(f"Region: {result.region}, Promedio de Victorias: {result.promedio_victorias}")

No se encontraron resultados para el tipo de juego especificado.


In [ ]:
# Control de Cupos (Torneos casi llenos)
stmt = (
    select(
        Tournament.name,
        Tournament.max_teams,
        func.count(TeamTournament.id_team).label("teams_registered")
    )
    .join(TeamTournament, isouter=True) # outer join por si no hay nadie aún
    .group_by(Tournament.id_tournament)
    .having(func.count(TeamTournament.id_team) < Tournament.max_teams)
)

rows = session.execute(stmt).all()
for result in rows:
    print(f"Tournament: {result.name}, Max Teams: {result.max_teams}, Teams Registered: {result.teams_registered}")

Tournament: Discussion Cup 32, Max Teams: 32, Teams Registered: 1
Tournament: Scientist Cup 114, Max Teams: 16, Teams Registered: 1
Tournament: Include Cup 85, Max Teams: 64, Teams Registered: 1
Tournament: Key Cup 121, Max Teams: 8, Teams Registered: 1
Tournament: Have Cup 49, Max Teams: 32, Teams Registered: 1
Tournament: Seem Cup 120, Max Teams: 64, Teams Registered: 1
Tournament: Answer Cup 68, Max Teams: 16, Teams Registered: 1
Tournament: Blood Cup 38, Max Teams: 8, Teams Registered: 1
Tournament: White Cup 126, Max Teams: 16, Teams Registered: 1
Tournament: Store Cup 147, Max Teams: 32, Teams Registered: 1
Tournament: Range Cup 24, Max Teams: 64, Teams Registered: 1
Tournament: Success Cup 17, Max Teams: 8, Teams Registered: 1
Tournament: Store Cup 90, Max Teams: 16, Teams Registered: 1
Tournament: Pay Cup 28, Max Teams: 16, Teams Registered: 1
Tournament: Seven Cup 140, Max Teams: 32, Teams Registered: 1
Tournament: Agency Cup 84, Max Teams: 32, Teams Registered: 1
Tournament: 